In [ ]:
pip install ffmpeg

  Preparing metadata (setup.py) ... done
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6083 sha256=7712b5fcf11beaa19a7425e7ee4589f3f0be523fe1c42f611b345ee1b9417803
  Stored in directory: /root/.cache/pip/wheels/26/21/0c/c26e09dff860a9071683e279445262346e008a9a1d2142c4ad
Successfully built ffmpeg


In [ ]:
from matplotlib.ticker import FuncFormatter

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from IPython.display import HTML

# --- Configuration ---
fps = 30                     # fixed playback speed
total_duration_sec = 15       # total animation time (without pause)
pause_duration_sec = 5       # pause at the end (in seconds)

# --- Data ---
import numpy as np

# --- Years ---
years = np.array([2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])

# --- Data (numeric values) ---
eigener_bereich = np.array([-4, 12, 56, 56, 44, 53, 56, 59])
gesamtbank      = np.array([-26, -27, -6, 2, -9, 25, 43, 41])
servicebank     = np.array([-23, -19, 10, 21, 7, 29, 53, 47])
kundenbank      = np.array([-26, -30, -13, -9, -18, 22, 36, 37])


# --- Auto-calculate smoothness for fixed 30 FPS ---
total_frames = total_duration_sec * fps
smooth_frames = total_frames // (len(years) - 1)
pause_frames = int(pause_duration_sec * fps)
total_frames_with_pause = total_frames + pause_frames

print(f"🎞️ FPS fixed at {fps}")
print(f"→ smooth_frames per year = {smooth_frames}, total frames = {total_frames}")

# --- Figure setup ---
fig, ax = plt.subplots(figsize=(15, 6))
# current figure and axes
fig.set_size_inches(15, 6)       # start size
ax.set_position([0.05, 0.1, 0.93, 0.83])  # [left, bottom, width, height]

# make figure smaller around same plot
fig.set_size_inches(13, 5)       # ✅ smaller outer face, same grid size


ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xlim(years[0] - 1, years[-1] + 1)
ax.set_ylim(-53, 70)
ax.set_yticks(np.arange(-50, 71, 20))
ax.grid(True, alpha=0.3, color="#404040")
# Add percent sign
# ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{int(y)}%"))

# Move labels away from the axis
ax.tick_params(axis="y", pad=10)
ax.tick_params(axis="x", pad=10)
ax.set_xticks(years)
fig.patch.set_facecolor("#eaeaef")  # overall figure background
ax.set_facecolor("#eaeaef")         # plot area background
ax.margins(x=0, y=0)
ax.tick_params(axis="both", length=0)  # hides tick lines on both x and y
ax.tick_params(axis="both", which="both", length=0)  # hide major & minor tick lines

# Keep grid lines at exact years (including 2025)
ax.set_xticks(years)
ax.grid(True, which="major", axis="x", color="#555557", linewidth=1)

# Shift labels between years
x_shifted = years - 0.5
ax.set_xticks(x_shifted, minor=True)
ax.set_xticklabels(years, minor=True)

# Hide major tick labels (at grid lines)
ax.tick_params(axis="x", which="major", bottom=False, labelbottom=False)
ax.tick_params(axis="x", which="minor", bottom=False, labelsize=10, pad=10, colors="#404040")

# Keep both frame edges visible
ax.spines["right"].set_visible(True)
ax.spines["top"].set_visible(False)

# --- Set x-limits to finish exactly at 2025 ---
ax.set_xlim(years[0] - 1, years[-1])   # ends grid exactly at 2025


# --- Hide extra frame edges ---
ax.spines["right"].set_visible(True)
ax.spines["top"].set_visible(False)

for spine in ax.spines.values():
    spine.set_color((0.25, 0.25, 0.25, 0.3))  # (R, G, B, alpha)
    spine.set_linewidth(1)



# Tick labels with alpha transparency
for tick in ax.xaxis.get_ticklabels():
    tick.set_color("#555557")

for tick in ax.yaxis.get_ticklabels():
    tick.set_color("#555557")

# --- Lines (static) ---
(line_eb,) = ax.plot([], [], marker="o", linewidth=3, color="#6A0D45",  # dark burgundy
                     markeredgewidth=4,
                     label="Eigener Bereich",linestyle="-",markersize=17,markerfacecolor="white",markeredgecolor="#6A0D45")

(line_gb,) = ax.plot([], [], marker="o", linewidth=2, color="#691A39",  # red striped style
                     markerfacecolor="white", markeredgewidth=2,
                     label="Gesamtbank",linestyle="-.",markersize=10,markeredgecolor="#E4C6C4")

(line_sb,) = ax.plot([], [], marker="o", linewidth=2, color="#E8AE96",  # light orange
                     markerfacecolor="white", markeredgewidth=2,
                     label="Servicebank",linestyle="--",markersize=10,alpha=0.9)

(line_kb,) = ax.plot([], [], marker="o", linewidth=2, color="#8F160D",  # dark red
                     markerfacecolor="white", markeredgewidth=2,
                     label="Kundenbank",linestyle="--",markersize=10,alpha=0.9)


# --- Moving checkpoint markers ---
# --- Points (for animation or highlighting latest values) ---
pt_eb = ax.plot([], [], marker="o", markersize=17, color="#6A0D45",markerfacecolor="white",markeredgewidth=4)[0]  # Eigener Bereich
pt_gb = ax.plot([], [], marker="o", markersize=10, color="#E4C6C4",markerfacecolor="white",markeredgewidth=2)[0]  # Gesamtbank
pt_sb = ax.plot([], [], marker="o", markersize=10, color="#E8AE96",markerfacecolor="white",markeredgewidth=2,alpha=0.9)[0]  # Servicebank
pt_kb = ax.plot([], [], marker="o", markersize=10, color="#8F160D",markerfacecolor="white",markeredgewidth=2,alpha=0.9)[0]  # Kundenbank


# ax.legend(
#     loc="lower right",
#     handlelength=3,     # longer line in legend
#     handletextpad=0.8,  # spacing between line and text
#     fontsize=10
# )
# year_text = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top", ha="left", fontsize=12)

# --- Init ---
def init():
    # Clear all line data
    for ln in (line_eb, line_gb, line_sb, line_kb):
        ln.set_data([], [])

    # Clear all point marker data
    for pt in (pt_eb, pt_gb, pt_sb, pt_kb):
        pt.set_data([], [])

    return (line_eb, line_gb, line_sb, line_kb,
            pt_eb, pt_gb, pt_sb, pt_kb)


# --- Animation ---
def animate(i):
    if i >= (len(years) - 1) * smooth_frames:
        i = (len(years) - 1) * smooth_frames - 1  # hold last frame during pause

    year_idx = i / smooth_frames
    lower = int(np.floor(year_idx))
    upper = min(lower + 1, len(years) - 1)
    frac = year_idx - lower


    # Interpolated x (shifted midpoint)
    x_now = x_shifted[lower] + frac * (x_shifted[upper] - x_shifted[lower])

    # Interpolated y values
    eb_now = eigener_bereich[lower] + frac * (eigener_bereich[upper] - eigener_bereich[lower])
    gb_now = gesamtbank[lower] + frac * (gesamtbank[upper] - gesamtbank[lower])
    sb_now = servicebank[lower] + frac * (servicebank[upper] - servicebank[lower])
    kb_now = kundenbank[lower] + frac * (kundenbank[upper] - kundenbank[lower])

    # Full x range for drawn portion (shifted)
    x_full = x_shifted[:upper + 1] if upper < len(years) - 1 else x_shifted

    # --- Update line data (matching lengths!) ---
    line_eb.set_data(np.append(x_full[:-1], x_now),
                     np.append(eigener_bereich[:upper], eb_now))
    line_gb.set_data(np.append(x_full[:-1], x_now),
                     np.append(gesamtbank[:upper], gb_now))
    line_sb.set_data(np.append(x_full[:-1], x_now),
                     np.append(servicebank[:upper], sb_now))
    line_kb.set_data(np.append(x_full[:-1], x_now),
                     np.append(kundenbank[:upper], kb_now))

    # --- Update point markers (shifted midpoints) ---
    pt_eb.set_data([x_now], [eb_now])
    pt_gb.set_data([x_now], [gb_now])
    pt_sb.set_data([x_now], [sb_now])
    pt_kb.set_data([x_now], [kb_now])



    return (line_eb, line_gb, line_sb, line_kb,
            pt_eb, pt_gb, pt_sb, pt_kb,)



# --- Frame timing ---
interval_ms = 1000 / fps  # milliseconds per frame


# --- Build animation ---
anim = FuncAnimation(
    fig, animate, init_func=init,
    frames=total_frames_with_pause, interval=interval_ms,
    blit=False, repeat=True
)
plt.draw()

# --- Save normal GIF ---
# --- Save GIF in high resolution ---
# gif_path = "weiterempfehlung_enps_2020_high_quality.gif"

# writer_gif = PillowWriter(fps=fps)

# anim.save(
#     gif_path,
#     writer=writer_gif,
#     dpi=200,                     # ✅ increase resolution (default ~100)
#     savefig_kwargs={
#         "facecolor": fig.get_facecolor(),  # ✅ preserve background color
#         "bbox_inches": "tight",            # ✅ trim whitespace
#         "pad_inches": 0                    # ✅ no extra padding
#     }
# )

# print(f"✅ Saved high-quality GIF: {gif_path}")

# --- Save MP4 ---
mp4_path = "weiterempfehlung_enps_anim.mp4"
writer_mp4 = FFMpegWriter(fps=fps, codec="libx264", bitrate=2500, extra_args=["-pix_fmt", "yuv420p"])
anim.save(mp4_path, writer=writer_mp4, dpi=300)
print(f"✅ Saved MP4: {mp4_path}")

plt.close(fig)

HTML(anim.to_html5_video())


🎞️ FPS fixed at 30
→ smooth_frames per year = 64, total frames = 450
✅ Saved MP4: weiterempfehlung_enps_anim.mp4


In [ ]:
from google.colab import files
files.download('weiterempfehlung_enps_anim.mp4')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# !ffmpeg -stream_loop 1 -i weiterempfehlung_enps_anim.mp4 -c copy enps.mp4


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab